# 02 — Profiling mémoire

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :
- mesurer la consommation mémoire avec `tracemalloc` ;
- identifier les allocations les plus coûteuses et leurs origines ;
- comparer des snapshots mémoire pour détecter les fuites ;
- utiliser `sys.getsizeof()` et `pympler` pour inspecter des objets ;
- découvrir `memray` pour du profiling mémoire avancé.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :
- le profiling CPU (`cProfile`, `line_profiler`) — notebook précédent ;
- le modèle mémoire de Python (tout est objet, compteur de références, GC) ;
- les `__slots__`, `weakref`, le garbage collector (section Métaprogrammation) ;
- les structures de données standard (list, dict, set, tuple).

## Plan

1. Pourquoi profiler la mémoire ?
2. `sys.getsizeof()` — taille d'un objet
3. `tracemalloc` — tracer les allocations
4. Snapshots et comparaisons
5. Filtrer par fichier ou module
6. `memray` — profiling mémoire avancé
7. Patterns de fuites mémoire courants
8. Synthèse
9. Exercices
10. Ressources

---

## 1. Pourquoi profiler la mémoire ?

| Symptôme | Cause possible |
|---|---|
| Process killed (OOM) | Allocation non bornée |
| Consommation croissante | Fuite mémoire (références conservées) |
| Lenteur progressive | Pression GC, swapping |
| Usage mémoire inattendu | Structures surdimensionnées |

Le profiling mémoire répond à : **qui alloue quoi, combien, et où dans le code ?**

---

## 2. `sys.getsizeof()` — taille d'un objet

`sys.getsizeof()` retourne la taille en octets d'un objet **seul** (sans compter les objets qu'il référence).

In [ ]:
import sys

In [ ]:
print(f"int(0)    : {sys.getsizeof(0)} octets")
print(f"int(42)   : {sys.getsizeof(42)} octets")
print(f"int(2**30): {sys.getsizeof(2**30)} octets")
print(f"int(2**60): {sys.getsizeof(2**60)} octets")

Les entiers Python ont une taille variable : plus le nombre est grand, plus il occupe de mémoire.

In [ ]:
print(f"str vide   : {sys.getsizeof('')} octets")
print(f"str 'abc'  : {sys.getsizeof('abc')} octets")
print(f"str 100 c  : {sys.getsizeof('a' * 100)} octets")

In [ ]:
print(f"list vide  : {sys.getsizeof([])} octets")
print(f"list 10 el : {sys.getsizeof(list(range(10)))} octets")
print(f"tuple vide : {sys.getsizeof(())} octets")
print(f"tuple 10 el: {sys.getsizeof(tuple(range(10)))} octets")

### Attention : `getsizeof` ne compte pas les contenus

Une liste de 1000 chaînes : `getsizeof` ne compte que le tableau de pointeurs, pas les chaînes elles-mêmes.

In [ ]:
lst = ["hello" * 100 for _ in range(1000)]
print(f"getsizeof(lst) = {sys.getsizeof(lst)} octets")
# Taille réelle bien plus grande — il faut compter les éléments
taille_reelle = sys.getsizeof(lst) + sum(sys.getsizeof(s) for s in lst)
print(f"Taille réelle ≈ {taille_reelle} octets ({taille_reelle / 1024:.1f} Ko)")

### Comparaison dict vs namedtuple vs __slots__

In [ ]:
from collections import namedtuple

class PointDict:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class PointSlots:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x = x
        self.y = y

PointNT = namedtuple("PointNT", ["x", "y"])

pd = PointDict(1.0, 2.0)
ps = PointSlots(1.0, 2.0)
pn = PointNT(1.0, 2.0)

print(f"PointDict    : {sys.getsizeof(pd)} + __dict__={sys.getsizeof(pd.__dict__)} octets")
print(f"PointSlots   : {sys.getsizeof(ps)} octets")
print(f"PointNT      : {sys.getsizeof(pn)} octets")

---

## 3. `tracemalloc` — tracer les allocations

`tracemalloc` est un module de la bibliothèque standard qui enregistre **chaque allocation mémoire** et sa traceback (pile d'appels).

In [ ]:
import tracemalloc

In [ ]:
tracemalloc.start()

# Allouer de la mémoire
data = [list(range(1000)) for _ in range(100)]

snapshot = tracemalloc.take_snapshot()
top_stats = snapshot.statistics("lineno")

print("[ Top 5 des allocations ]")
for stat in top_stats[:5]:
    print(stat)

tracemalloc.stop()

Chaque ligne indique :
- le **fichier et numéro de ligne** qui a alloué ;
- la **taille totale** allouée depuis cette ligne ;
- le **nombre de blocs** alloués.

### Grouper par fichier

In [ ]:
tracemalloc.start()

data = {f"clé_{i}": list(range(500)) for i in range(200)}

snapshot = tracemalloc.take_snapshot()
stats = snapshot.statistics("filename")

print("[ Top 5 par fichier ]")
for stat in stats[:5]:
    print(stat)

tracemalloc.stop()

### Tracer avec la pile d'appels complète

In [ ]:
tracemalloc.start(25)  # 25 frames max dans la traceback

def creer_donnees():
    return [bytearray(1024) for _ in range(100)]

def pipeline():
    return creer_donnees()

data = pipeline()
snapshot = tracemalloc.take_snapshot()
stats = snapshot.statistics("traceback")

print("[ Allocation la plus coûteuse — traceback complète ]")
print(stats[0].traceback.format())

tracemalloc.stop()

---

## 4. Snapshots et comparaisons

La force de `tracemalloc` est de **comparer deux snapshots** pour détecter les fuites : les allocations qui augmentent entre deux points du programme.

In [ ]:
tracemalloc.start()

# Snapshot initial
donnees = []
snap1 = tracemalloc.take_snapshot()

# Simuler une accumulation
for _ in range(50):
    donnees.append(bytearray(10_000))

snap2 = tracemalloc.take_snapshot()

# Comparer
diff = snap2.compare_to(snap1, "lineno")
print("[ Top 5 des augmentations ]")
for stat in diff[:5]:
    print(stat)

tracemalloc.stop()

Les lignes avec `+` indiquent les allocations **nouvelles** entre les deux snapshots. C'est exactement ce qu'on cherche pour détecter une fuite.

### Surveiller l'évolution dans le temps

In [ ]:
tracemalloc.start()

snapshots = []
cache = {}

for i in range(5):
    # Simuler l'accumulation d'un cache qui grossit
    cache[i] = bytearray(50_000)
    snapshots.append(tracemalloc.take_snapshot())

# Afficher l'évolution
for i in range(1, len(snapshots)):
    diff = snapshots[i].compare_to(snapshots[i-1], "lineno")
    total = sum(s.size_diff for s in diff if s.size_diff > 0)
    print(f"Étape {i}: +{total / 1024:.1f} Ko")

tracemalloc.stop()

---

## 5. Filtrer par fichier ou module

`tracemalloc` fournit des filtres pour se concentrer sur **votre code** et ignorer les allocations de la bibliothèque standard.

In [ ]:
tracemalloc.start()

import json as json_mod
data = json_mod.loads('[' + ','.join(str(i) for i in range(1000)) + ']')
more = list(range(10_000))

snapshot = tracemalloc.take_snapshot()

# Filtrer : exclure les modules de la stdlib
filtered = snapshot.filter_traces([
    tracemalloc.Filter(False, "<frozen *>"),
    tracemalloc.Filter(False, "<unknown>"),
])

for stat in filtered.statistics("lineno")[:5]:
    print(stat)

tracemalloc.stop()

---

## 6. `memray` — profiling mémoire avancé

`memray` est un profiler mémoire développé par Bloomberg, plus puissant que `tracemalloc` :
- visualisation interactive (flame graphs mémoire, tree maps) ;
- suivi des allocations C et Python ;
- détection de fuites natives.

> **Installation :** `pip install memray`

### Utilisation en ligne de commande

```bash
# Enregistrer un profil
memray run -o output.bin mon_script.py

# Générer un flame graph HTML
memray flamegraph output.bin -o flamegraph.html

# Rapport textuel des fuites
memray stats output.bin

# Tree map interactif
memray tree output.bin
```

### Utilisation dans un script

```python
import memray

with memray.Tracker("output.bin"):
    # Code à profiler
    data = [bytearray(1024) for _ in range(10_000)]
```

### `memray` vs `tracemalloc`

| Critère | `tracemalloc` | `memray` |
|---|---|---|
| Installation | Standard | `pip install memray` |
| Allocations C | Non | Oui |
| Visualisation | Texte | HTML (flame graph, tree map) |
| Surcoût | Modéré | Modéré |
| Détection de fuites | Manuelle (diff snapshots) | Automatique (`memray stats`) |
| Multiplateformes | Oui | Linux / macOS |

---

## 7. Patterns de fuites mémoire courants

### 7.1. Cache non borné

In [ ]:
# MAUVAIS — le cache grossit sans limite
cache = {}

def traiter(clé):
    if clé not in cache:
        cache[clé] = bytearray(1024)  # simule un résultat coûteux
    return cache[clé]

# Après 100 000 appels avec des clés uniques : 100 Mo de cache !

In [ ]:
# BON — utiliser functools.lru_cache avec maxsize
from functools import lru_cache

@lru_cache(maxsize=1024)
def traiter_borne(clé: int) -> bytes:
    return bytes(1024)  # simule un résultat coûteux

print(f"Cache info : {traiter_borne.cache_info()}")

### 7.2. Références circulaires

In [ ]:
import gc

class Noeud:
    def __init__(self, valeur):
        self.valeur = valeur
        self.voisin = None

a = Noeud("A")
b = Noeud("B")
a.voisin = b
b.voisin = a  # référence circulaire

del a, b
# Les objets ne sont pas libérés immédiatement (compteur de ref > 0)
# Il faut attendre le GC
gc.collect()
print(f"Objets collectés : le GC gère les cycles, mais avec un coût")

### 7.3. Closures capturant de gros objets

In [ ]:
def creer_handler():
    gros_objet = bytearray(10_000_000)  # 10 Mo
    def handler():
        return len(gros_objet)  # capture gros_objet par référence
    return handler

h = creer_handler()
# gros_objet est maintenu en vie tant que h existe !
print(f"Taille capturée : {sys.getsizeof(h.__closure__[0].cell_contents)} octets")

### 7.4. Listes accumulées dans des attributs de classe

In [ ]:
class Evenement:
    historique = []  # DANGER : partagé entre toutes les instances !

    def __init__(self, nom):
        self.nom = nom
        self.historique.append(self)  # ne sera jamais libéré

# Chaque instance reste en mémoire via historique
for i in range(1000):
    Evenement(f"evt_{i}")

print(f"Instances en mémoire : {len(Evenement.historique)}")

---

## 8. Synthèse

| Outil | Usage | Niveau |
|---|---|---|
| `sys.getsizeof(obj)` | Taille d'un seul objet (sans contenus) | Rapide |
| `tracemalloc` | Traçage des allocations avec traceback | Standard |
| `tracemalloc` snapshots | Comparaison avant/après pour détecter les fuites | Standard |
| `memray` | Profiling avancé avec visualisation | Avancé |

**Règles à retenir :**
- `sys.getsizeof()` ne mesure qu'un objet, pas ses contenus — ne pas confondre avec la taille réelle.
- `tracemalloc` + diff de snapshots est la technique de base pour détecter les fuites.
- Les fuites Python viennent des **références conservées** (caches, closures, attributs de classe), pas des oublis de `free()`.
- `__slots__` réduit la consommation mémoire de 30-50 % sur les petits objets.
- Utilisez `lru_cache(maxsize=...)` plutôt qu'un dict libre comme cache.

---

## 9. Exercices

### Exercice 1 — Taille réelle récursive *(facile)*

Écrire une fonction `taille_profonde(obj)` qui calcule la taille totale d'un objet, y compris ses contenus (pour les listes, dicts, tuples, sets). Utilisez `sys.getsizeof()` et la récursion.

```python
data = {"a": [1, 2, 3], "b": "hello"}
print(taille_profonde(data))
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Profiling_memoire", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import sys

def taille_profonde(obj, vus=None):
    if vus is None:
        vus = set()
    obj_id = id(obj)
    if obj_id in vus:
        return 0
    vus.add(obj_id)
    taille = sys.getsizeof(obj)
    if isinstance(obj, dict):
        for k, v in obj.items():
            taille += taille_profonde(k, vus) + taille_profonde(v, vus)
    elif isinstance(obj, (list, tuple, set, frozenset)):
        for item in obj:
            taille += taille_profonde(item, vus)
    return taille

data = {"a": [1, 2, 3], "b": "hello"}
print(f"Taille profonde : {taille_profonde(data)} octets")
```

</details>

### Exercice 2 — Détecter une fuite avec `tracemalloc` *(moyen)*

Le code suivant a une fuite mémoire. Utilisez `tracemalloc` avec deux snapshots pour la localiser :

```python
registre = []

def enregistrer_evenement(nom, donnees):
    registre.append({"nom": nom, "donnees": donnees, "copie": donnees.copy()})

for i in range(1000):
    enregistrer_evenement(f"evt_{i}", list(range(500)))
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Profiling_memoire", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import tracemalloc

tracemalloc.start()
snap1 = tracemalloc.take_snapshot()

registre = []

def enregistrer_evenement(nom, donnees):
    registre.append({"nom": nom, "donnees": donnees, "copie": donnees.copy()})

for i in range(1000):
    enregistrer_evenement(f"evt_{i}", list(range(500)))

snap2 = tracemalloc.take_snapshot()

diff = snap2.compare_to(snap1, "lineno")
print("[ Top 5 augmentations ]")
for stat in diff[:5]:
    print(stat)

# La fuite est dans registre.append() : chaque événement stocke
# les données ET une copie. La liste registre grossit sans limite.
tracemalloc.stop()
```

</details>

### Exercice 3 — Comparer les structures *(moyen)*

Mesurez et comparez la consommation mémoire de 100 000 « points (x, y) » stockés sous quatre formes :
1. `dict` : `{"x": val, "y": val}`
2. `namedtuple`
3. classe avec `__slots__`
4. `tuple` brut `(x, y)`

Affichez un tableau comparatif.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Profiling_memoire", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import sys
from collections import namedtuple

class PointSlots:
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x = x
        self.y = y

PointNT = namedtuple("PointNT", ["x", "y"])

N = 100_000

structures = {
    "dict": [{"x": i, "y": i} for i in range(N)],
    "namedtuple": [PointNT(i, i) for i in range(N)],
    "__slots__": [PointSlots(i, i) for i in range(N)],
    "tuple": [(i, i) for i in range(N)],
}

for nom, lst in structures.items():
    taille = sys.getsizeof(lst) + sum(sys.getsizeof(obj) for obj in lst)
    print(f"{nom:15s} : {taille / 1024 / 1024:.2f} Mo")
```

</details>

### Exercice 4 — Moniteur mémoire continu *(difficile)*

Écrire un décorateur `@surveiller_memoire` qui :
1. Prend un snapshot `tracemalloc` avant l'appel ;
2. Prend un snapshot après ;
3. Affiche la différence si elle dépasse un seuil (paramètre `seuil_ko`).

```python
@surveiller_memoire(seuil_ko=100)
def gros_traitement(n):
    return [bytearray(1024) for _ in range(n)]
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Profiling_memoire", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import tracemalloc
import functools

def surveiller_memoire(seuil_ko: float = 100):
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            deja_actif = tracemalloc.is_tracing()
            if not deja_actif:
                tracemalloc.start()
            snap1 = tracemalloc.take_snapshot()

            result = fn(*args, **kwargs)

            snap2 = tracemalloc.take_snapshot()
            diff = snap2.compare_to(snap1, "lineno")
            total_ko = sum(s.size_diff for s in diff if s.size_diff > 0) / 1024

            if total_ko > seuil_ko:
                print(f"[MEMOIRE] {fn.__name__} : +{total_ko:.1f} Ko")
                for stat in diff[:3]:
                    print(f"  {stat}")

            if not deja_actif:
                tracemalloc.stop()
            return result
        return wrapper
    return decorator

@surveiller_memoire(seuil_ko=50)
def gros_traitement(n):
    return [bytearray(1024) for _ in range(n)]

resultat = gros_traitement(500)
```

</details>

---

## 10. Ressources

- [Module `tracemalloc` — documentation officielle](https://docs.python.org/3/library/tracemalloc.html)
- [Module `sys` — `getsizeof`](https://docs.python.org/3/library/sys.html#sys.getsizeof)
- [`memray` — GitHub Bloomberg](https://github.com/bloomberg/memray)
- [Memory Management in CPython](https://realpython.com/python-memory-management/)